In [ ]:
import requests
import bs4
from tqdm import tqdm
import json

TOTAL_PAGES = 335


def get_url_for_page(n: int):
    return f"https://community.chocolatey.org/packages?sortOrder=package-download-count&page={n}&prerelease=False&moderatorQueue=False&moderationStatus=all-statuses"


def get_and_parse(url: str):
    res = requests.get(url)
    return bs4.BeautifulSoup(res.content)


def find_packages(content):
    return content.select(".package-list-view")


def get_install_commands(url):
    return [
        input_el.attrs["value"]
        for input_el in find_packages(get_and_parse(url))[0].find_all("input")
    ]


urls = [get_url_for_page(i) for i in range(1, TOTAL_PAGES)]
commands = [
    cmd
    for url in tqdm(urls, desc="Finding installation commands")
    for cmd in get_install_commands(url)
]


Finding installation commands: 100%|██████████| 334/334 [02:21<00:00,  2.36it/s]


In [ ]:
commands = [cmd.split(" ")[-1] for cmd in commands]
with open("install-commands.json", "w") as file:
    json.dump(commands, file)